# MultiMNIST comparison — stronger Ours tuning, existing baselines reused

**Run cells 1–10 in order on an A100.** This notebook trains only Entropic LMO-MGDA. It reads all six completed baselines from the existing `multimnist_a100_v2` experiment. No baseline training commands are present. Ablations are in `MultiMNIST_Ablations_A100.ipynb`.

The previous three-seed means were: MGDA 84.38, MGDA + Muon 85.45, FAMO 86.27, FAMO + Muon 87.65, Muon (LS) 87.75, MOON 87.47, and Ours 87.20. These are existing local results, not the paper's published numbers.

The old search tested 8 LR/momentum combinations on one seed with a fixed eta. This search explores **96 initial LR/momentum/schedule combinations**, then refines eta for four leading configurations, and confirms up to four candidates plus the old configuration on **three validation seeds**. The final method remains spectral + entropic + blended; no update equation or architecture is changed.

| Search component | Values / rule |
| --- | --- |
| Model LR | 0.001, 0.003, 0.006, 0.01, 0.015, 0.022, 0.03, 0.045 |
| Momentum injection alpha | 0.02, 0.05, 0.1, 0.2, 0.5, 0.8 |
| LR schedule | Original reference schedule; cosine decay to 5% of initial LR |
| Initial screening | 25 epochs for all; top 24 continue to 60; top 8 continue to 100 |
| Eta refinement | 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 3e-3, 1e-2; top 4 LR/momentum/schedule families; screen at 60, top 8 to 100 |
| Confirmation | Top 4 candidates plus the old configuration, seeds 42/43/44, 100 epochs |
| Selection score | Mean across validation seeds of accuracy averaged over epochs 96–100; lower seed std breaks ties |

The old configuration is protected from pruning. Its matching old validation checkpoints are reused when available. Every trial uses a fixed split of **9,000 training / 1,000 validation images** from the training set. The search never reads test images or baseline scores. After selection, three fresh models train on all 10,000 training images and are evaluated on the same test set as the existing baselines, at epoch 100.

Cosine decay is a new training hyperparameter for Ours, not a setting claimed to come from MOON. The first five eta values follow [MOON Table 8](https://arxiv.org/pdf/2608.11749v1); the two larger values extend our own search. Baseline settings stay as previously run from the [released reference code](https://github.com/KunlinLyu/MOON/tree/37319d14765595577b9b01fe91d6fcfca5ffbd7c).

**Budget:** roughly 6,000–7,000 training epoch-equivalents before cache reuse, plus 300 final-training epochs. Allow several hours on A100. Run the three search stages in separate sessions if needed. A rerun resumes or skips existing validation trials and never discards completed work.

Selection is based on validation; no first-place test result is guaranteed. The test set has already been inspected during experiment development. The report records the expanded Ours-only search; it is not an equal-search-budget comparison.

In [ ]:
#@title 1. Existing experiment and new Ours search
REPO_URL = "https://github.com/alirezamirrokni/LMO-MOO.git"
REPO_COMMIT = "ec912121ed084551b9898caee4334ebec3445474"
EXPERIMENT_NAME = "multimnist_a100_v2" #@param {type:"string"}
SEARCH_NAME = "ours_retune_v3" #@param {type:"string"}
DATA_CACHE_EXPERIMENT = "multimnist_a100_v1"
SEEDS = [42, 43, 44]
EPOCHS = 100
BATCH_SIZE = 256

In [ ]:
#@title 2. Check A100, mount Drive and enable live console logs
import os, sys, json, subprocess, shutil, time, hashlib, zipfile, csv
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME, f"Current GPU: {GPU_NAME}. Select A100 and reconnect."
print("GPU:", GPU_NAME, "| PyTorch:", torch.__version__)
drive.mount("/content/drive")
assert EXPERIMENT_NAME and Path(EXPERIMENT_NAME).name == EXPERIMENT_NAME and EXPERIMENT_NAME not in {".", ".."}
DRIVE_ROOT = Path("/content/drive/MyDrive/LMO-MOO")
BASE_ROOT = DRIVE_ROOT / EXPERIMENT_NAME
assert SEARCH_NAME and Path(SEARCH_NAME).name == SEARCH_NAME and SEARCH_NAME not in {".", ".."}
RUN_ROOT = BASE_ROOT / SEARCH_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
BASELINE_ROOT = BASE_ROOT / "results" / "multimnist"
TUNING_ROOT = RUN_ROOT / "tuning"
SELECTION_FILE = TUNING_ROOT / "selection.json"
INCUMBENT_SELECTION = BASE_ROOT / "tuning" / "selection.json"
OUTPUT_ROOT = RUN_ROOT / "results" / "multimnist"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/LMO-MOO-retune-v3")
DATA = Path("/content/LMO-MOO-data/multimnist")
print("Persistent outputs:", OUTPUT_ROOT)


import codecs, signal, shlex, uuid
LOG_DIR = RUN_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

def run_live(cmd, *, cwd=None, label="process"):
    """Forward stdout/stderr chunks immediately, preserving carriage returns.

    Raw console logs are also saved to Drive. An interrupted cell stops the
    entire subprocess group, including children launched by the suite.
    """
    cmd = list(map(str, cmd))
    print("$ " + shlex.join(cmd), flush=True)
    safe_label = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)
    log_path = LOG_DIR / (time.strftime("%Y%m%d_%H%M%S") + "_" + safe_label + "_" + uuid.uuid4().hex[:6] + ".log")
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    print("Console log:", log_path, flush=True)
    with log_path.open("wb", buffering=0) as log:
        process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, bufsize=0,
                                   start_new_session=True)
        decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
        try:
            while True:
                chunk = os.read(process.stdout.fileno(), 8192)
                if not chunk:
                    break
                # Display first so a slow Drive write cannot hide current output.
                sys.stdout.write(decoder.decode(chunk))
                sys.stdout.flush()
                log.write(chunk)
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=5)
                except (subprocess.TimeoutExpired, KeyboardInterrupt):
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
            raise
        finally:
            process.stdout.close()
    print(f"\n[{label}] Exit code: {returncode}", flush=True)
    if returncode:
        raise subprocess.CalledProcessError(returncode, cmd)
    return log_path

In [ ]:
#@title 3. Load the pinned code and install dependencies
if not REPO.exists():
    run_live(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "--sparse", "--progress", REPO_URL, REPO], label="clone")
else:
    assert (REPO / ".git").exists(), f"{REPO} is not a Git checkout."
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty, "Save local tracked-file edits before rerunning setup."
if subprocess.run(["git", "-C", str(REPO), "cat-file", "-e", REPO_COMMIT + "^{commit}"], capture_output=True).returncode:
    run_live(["git", "-C", REPO, "fetch", "origin", REPO_COMMIT], label="fetch")
run_live(["git", "-C", REPO, "sparse-checkout", "set", "experiments", "methods", "scripts"], label="checkout-files")
run_live(["git", "-C", REPO, "checkout", "--detach", REPO_COMMIT], label="checkout-revision")
# Preserve Colab's installed CUDA-compatible torch and torchvision pair.
run_live([sys.executable, "-m", "pip", "install", "-r", REPO / "requirements-modern.txt", "pandas"], label="dependencies")
run_live([sys.executable, "-u", "-c", "import torch, torchvision; from experiments.multimnist.trainer import parser; print('Imports OK:', torch.__version__, torchvision.__version__)"], cwd=REPO, label="import-check")
os.chdir(REPO)
def run_script(filename, *args):
    return run_live([sys.executable, "-u", REPO / filename, *args], cwd=REPO, label=Path(filename).stem)

def atomic_json(path, value):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2) + "\n")
    tmp.replace(path)

identity = {"repo": REPO_URL, "commit": REPO_COMMIT, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "seeds": SEEDS, "dataset_seed": 2026,
            "train_samples": 10000, "test_samples": 1000,
            "baseline_root": str(BASELINE_ROOT)}
manifest = RUN_ROOT / "workflow.json"
if manifest.exists():
    assert json.loads(manifest.read_text()) == identity, "Workflow settings changed. Use a new SEARCH_NAME."
else:
    atomic_json(manifest, identity)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_ROOT / ("environment_" + time.strftime("%Y%m%d_%H%M%S") + ".txt")).write_text(versions)
print("Pinned code revision:", REPO_COMMIT)
print("Existing baselines are read from:", BASELINE_ROOT)
print("New Ours outputs are written to:", RUN_ROOT)

In [ ]:
#@title 4. Prepare and cache the fixed dataset
CACHE = DRIVE_ROOT / DATA_CACHE_EXPERIMENT / "data"
CACHE.mkdir(parents=True, exist_ok=True)
ARCHIVE = CACHE / "multimnist_seed2026.zip"
LOCAL_ARCHIVE = Path("/content/multimnist_seed2026.zip")
if not ARCHIVE.exists():
    # Remove only a partial generated dataset from an interrupted preparation.
    if DATA.exists():
        shutil.rmtree(DATA)
    run_script("prepare_multimnist.py", "--download", "--output", DATA,
               "--mnist-root", "/content/LMO-MOO-data/mnist",
               "--train-samples", 10000, "--test-samples", 1000, "--seed", 2026)
    with zipfile.ZipFile(LOCAL_ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(DATA.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(DATA))
    print("Saving the dataset archive to Drive...", flush=True)
    partial = ARCHIVE.with_suffix(".zip.partial")
    shutil.copy2(LOCAL_ARCHIVE, partial)
    partial.replace(ARCHIVE)
else:
    print("Restoring the dataset archive from Drive...", flush=True)
    shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)

digest = hashlib.sha256(LOCAL_ARCHIVE.read_bytes()).hexdigest()
hash_file = CACHE / "dataset_archive.sha256"
if hash_file.exists():
    assert hash_file.read_text().strip() == digest, "Dataset archive checksum mismatch."
else:
    hash_file.write_text(digest + "\n")
# Reuse local data within a session; restore it after reconnecting.
marker = DATA.parent / "archive.sha256"
if not (DATA.exists() and marker.exists() and marker.read_text().strip() == digest):
    if DATA.exists():
        shutil.rmtree(DATA)
    DATA.mkdir(parents=True)
    with zipfile.ZipFile(LOCAL_ARCHIVE) as z:
        assert z.testzip() is None, "Damaged dataset archive."
        for info in z.infolist():
            target = (DATA / info.filename).resolve()
            assert target.is_relative_to(DATA.resolve()), "Unsafe archive entry."
        for info in z.infolist():
            z.extract(info, DATA)
    marker.write_text(digest + "\n")
for split, expected in [("train", 10000), ("test", 1000)]:
    with (DATA / split / "labels.csv").open() as f:
        rows = list(csv.reader(f))
    assert len(rows) == expected
    assert all((DATA / split / "2" / row[0]).is_file() for row in rows)
    print(split, len(rows), "images")
print("Dataset ready on local disk:", DATA)

In [ ]:
#@title 5. Verify all 18 baseline runs and prepare the Ours-only search
from IPython.display import display
import pandas as pd
import importlib
# A live Colab kernel may still cache packages from the previous checkout.
active_repo = REPO.resolve()
active_commit = subprocess.check_output(
    ["git", "-C", str(active_repo), "rev-parse", "HEAD"], text=True).strip()
if active_commit != REPO_COMMIT:
    raise RuntimeError("The active checkout is not the pinned revision. Rerun cells 1–4 of this notebook.")
sys.path[:] = [entry for entry in sys.path if entry != str(active_repo)]
sys.path.insert(0, str(active_repo))
for module_name in list(sys.modules):
    if module_name in {"experiments", "methods"} or module_name.startswith(("experiments.", "methods.")):
        del sys.modules[module_name]
importlib.invalidate_caches()
from experiments.multimnist import trainer as active_trainer
if Path(active_trainer.__file__).resolve() != active_repo / "experiments/multimnist/trainer.py":
    raise RuntimeError("Python imported a different checkout. Restart the session and rerun cells 1–5.")
training_parser = active_trainer.parser
run_signature = active_trainer.run_signature
data_fingerprint = active_trainer.data_fingerprint
print("Training module:", active_trainer.__file__)
COMMON = ["--data-path", DATA, "--device", "cuda:0", "--epochs", EPOCHS,
          "--batch-size", BATCH_SIZE, "--workers", 0, "--threads", 4, "--cache-data"]
DATA_SHA256 = data_fingerprint(DATA)
TRAIN_SHA256 = data_fingerprint(DATA, splits=("train",))

def settings_flags(settings):
    keys = ["lr", "eta", "alpha", "oracle", "weights", "momentum", "lr_schedule", "min_lr_ratio"]
    return [part for k in keys if k in settings for part in ("--"+k.replace("_", "-"), settings[k])]

def load_selected(path):
    selection = json.loads(path.read_text())
    assert selection["test_used"] is False and selection["settings"]["selection"] == "validation"
    assert selection["settings"]["train_sha256"] == TRAIN_SHA256, "Selection used different images."
    selected = selection["selected"]
    assert selected["epochs"] == EPOCHS and selected["batch_size"] == BATCH_SIZE
    return selection, selected

def training_args(root, tag, method, seed, flags):
    return ["--output-root", root, "--tag", tag, "--method", method, "--seed", seed,
            *COMMON, *flags, "--resume"]

def status_for(root, tag, method, seed, flags):
    directory = root / tag / method / f"seed{seed}"
    args = training_parser().parse_args(list(map(str, training_args(root, tag, method, seed, flags))))
    expected = run_signature(args, DATA_SHA256)
    for name in ("config.json", "summary.json"):
        path = directory / name
        if path.exists() and json.loads(path.read_text()).get("signature") != expected:
            raise ValueError(f"Configuration/data mismatch: {path}")
    result = dict(tag=tag, method=method, seed=seed, status="pending", epoch=0)
    summary_path = directory / "summary.json"
    if summary_path.exists():
        s = json.loads(summary_path.read_text())
        assert s["seed"] == seed and s["method"] == method and s["tag"] == tag
        assert s.get("selection") == "test" and not s.get("smoke")
        assert 0 <= s["epoch"] <= EPOCHS
        complete = s.get("completed") is True and s["epoch"] == EPOCHS
        result.update(status="complete" if complete else "resume", epoch=s["epoch"])
        if not complete and not (directory / "checkpoint.pt").exists():
            raise FileNotFoundError(f"Partial run has no checkpoint: {directory}")
    return result

def run_ours_jobs(root, jobs):
    for tag, flags in jobs:
        for seed in SEEDS:
            status = status_for(root, tag, "ours", seed, flags)
            print(f"{tag}/ours/seed{seed}: {status['status']}", flush=True)
            if status["status"] == "complete":
                continue
            run_script("run_multimnist.py", *training_args(root, tag, "ours", seed, flags))
            assert status_for(root, tag, "ours", seed, flags)["status"] == "complete"

BASELINE_CONFIGS = {
    "moon": ["--lr", 0.001, "--weight-lr", 0.0001, "--gamma", 0.001],
    "famo": ["--lr", 0.001, "--weight-lr", 0.025, "--gamma", 0.01],
    "famo_muon": ["--lr", 0.001, "--weight-lr", 0.025, "--gamma", 0.01],
    "mgda": ["--lr", 0.001], "mgda_muon": ["--lr", 0.001], "muon_ls": ["--lr", 0.001],
}
baseline_status = pd.DataFrame([status_for(BASELINE_ROOT, "main", method, seed, flags)
    for method, flags in BASELINE_CONFIGS.items() for seed in SEEDS])
display(baseline_status)
assert (baseline_status.status == "complete").all(), "Locate all completed baselines in EXPERIMENT_NAME; this notebook will not retrain them."
old_selection, OLD_SELECTED = load_selected(INCUMBENT_SELECTION)
print("Old validation-selected configuration:", OLD_SELECTED)
SEARCH_ARGS = ["--data-path", DATA, "--output-root", TUNING_ROOT,
               "--incumbent-selection", INCUMBENT_SELECTION,
               "--reuse-root", BASE_ROOT / "tuning", "--device", "cuda:0", "--threads", 4]
atomic_json(RUN_ROOT / "baseline_reuse.json", {"root": str(BASELINE_ROOT), "data_sha256": DATA_SHA256,
            "methods": list(BASELINE_CONFIGS), "seeds": SEEDS, "training_performed": False})
print("All baselines verified. Cells 6–8 tune only Ours on validation.")

In [ ]:
#@title 6. Search model LR, momentum injection and LR schedule
run_script("run_multimnist_search.py", *SEARCH_ARGS, "--through", "schedule")

In [ ]:
#@title 7. Refine the entropic weight step eta
run_script("run_multimnist_search.py", *SEARCH_ARGS, "--through", "eta")

In [ ]:
#@title 8. Confirm on three validation seeds and freeze the selection
run_script("run_multimnist_search.py", *SEARCH_ARGS, "--through", "confirm")
selection, SELECTED = load_selected(SELECTION_FILE)
print("Selected settings:", json.dumps(SELECTED, indent=2))
print("Three-seed validation score:", selection["validation"]["score"])
print("Old configuration on the same validation seeds:", selection["incumbent_validation"]["score"])
display(pd.DataFrame([{"tag": r["tag"], "validation_score": r["score"], "seed_std": r["score_std"],
                      "final_epoch_avg": r["final_avg"]} for r in selection["confirmation"]]))

In [ ]:
#@title 9. Train only the selected Ours configuration on the full training set
selection, SELECTED = load_selected(SELECTION_FILE)
freeze = RUN_ROOT / "final_configuration.json"
if freeze.exists():
    assert json.loads(freeze.read_text()) == SELECTED, "Selection changed; use a new SEARCH_NAME."
else:
    atomic_json(freeze, SELECTED)
OURS_FLAGS = settings_flags(SELECTED)
# OUTPUT_ROOT is new; old Ours and every baseline remain in BASELINE_ROOT.
run_ours_jobs(OUTPUT_ROOT, [("main", OURS_FLAGS)])

In [ ]:
#@title 10. Report the comparison using saved baselines and newly selected Ours
REPORT = RUN_ROOT / "comparison_report"
run_script("report_multimnist.py", "--root", BASELINE_ROOT, "--ours-root", OUTPUT_ROOT,
           "--section", "comparison", "--baseline-source", "reproduced",
           "--seeds", *SEEDS, "--out", REPORT)
results = pd.read_csv(REPORT / "results.csv")
display(results[["method", "n_seeds", "left", "right", "avg", "avg_std"]])
assert len(results) == 7 and (results.n_seeds == 3).all(), "Some final runs are incomplete."
# Record the selected training recipe alongside the table. Do not select a run by its test score.
selection = json.loads(SELECTION_FILE.read_text())
atomic_json(REPORT / "ours_search_protocol.json", {"settings": selection["settings"],
            "selected": selection["selected"], "criterion": selection["criterion"],
            "baseline_search": "Existing released-code settings; no additional baseline tuning"})
BUNDLE = RUN_ROOT / "multimnist_comparison_reports.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    paths = list(REPORT.glob("*")) + [SELECTION_FILE, TUNING_ROOT / "search.json", RUN_ROOT / "final_configuration.json"]
    for method in BASELINE_CONFIGS:
        for seed in SEEDS:
            folder = BASELINE_ROOT / "main" / method / f"seed{seed}"
            paths += [folder / "config.json", folder / "summary.json"]
    for seed in SEEDS:
        folder = OUTPUT_ROOT / "main" / "ours" / f"seed{seed}"
        paths += [folder / "config.json", folder / "summary.json"]
    for path in paths:
        if path.is_file(): archive.write(path, path.relative_to(BASE_ROOT))
print("Comparison LaTeX:", REPORT / "table.tex")
print("Measured CSV:", REPORT / "results.csv")
print("Download the report bundle from Drive:", BUNDLE)

## Continue after a disconnection

Run **1–5**, then the interrupted stage (**6**, **7**, or **8**). Each later search stage checks and reuses earlier stages automatically. After selection exists, run **9–10**; these final cells never launch a new search. Keep `EXPERIMENT_NAME` and `SEARCH_NAME` unchanged.

All new files are under `MyDrive/LMO-MOO/multimnist_a100_v2/ours_retune_v3/`. Old Ours results and completed baseline files stay in their original folders. This notebook contains no ablation runs. For those, open the separate [ablation notebook](https://colab.research.google.com/github/alirezamirrokni/LMO-MOO/blob/main/notebooks/MultiMNIST_Ablations_A100.ipynb) after finishing the comparison.

Logs stream live, one training summary per epoch. Do not run both notebooks simultaneously against the same output folders. Disconnect and delete the runtime when finished; Drive checkpoints remain available.